# Figure P01 — Phage: structural entropy predicts annotation, ranks missing genes, and corrects an annotation

**Status:** draft (Setonix integrator, agent-handoffs#24). Provisional ID; the figure number is set in `figure_plan.md` (proposed main Fig. 6).

**Caption (draft).** Phage genomes from Inphared (Pharokka + phold annotation); six-frame ORFs ≥90 aa; an ORF is a gene if it matches an annotated CDS on the same strand and phase (≥90% overlap, ≥98% amino-acid identity), using the corrected (v2) matcher.
**(a)** Mutual information between each per-ORF variable and the annotation label (class-balanced sample, 20 bins per axis; bars, bootstrap 95% CI). Numbers give the share of each variable's information that survives conditioning on length (20 quantile strata, full population). 36,025 genomes, 6,754,361 ORFs, 39.8% genes.
**(b)** Grouped cross-validated ranking models (XGBoost random forest; 31,200 genomes, 5,887,690 ORFs), ROC AUC and average precision on corrected labels. The legacy model, trained on labels that missed every minus-strand gene, learned "plus strand = gene" and collapses on corrected labels. Dotted line: prevalence (0.40), the average precision of a random ranking.
**(c)** Full-length (query and target coverage ≥0.8) Foldseek structural hits for 20,000 ORFs per arm (4,102 / 4,039 for replacement pairs) against Swiss-Prot, PDB100, CATH50 and BFVD (any), or excluding the viral BFVD; Wilson 95% CI. Controls are low-scoring antisense, frameshift and intergenic ORFs and the candidates' own 3Di strings shuffled.
**(d)** Phage OM869708 (5,753 bp). Annotated CDS (grey) and all ORFs ≥90 aa by frame, coloured by the corrected model's coding probability; outlined ORFs match an annotated CDS. ORF3 (outlined in red; p = 0.89) overlaps the annotated CDS_0007 / ORF7 (p = 0.18) in another frame and is ranked 179th of 4,102 replacement candidates.

**Manuscript text:** Results, phage section (to be written, checkpoint D).

## Provenance

| item | value |
|---|---|
| plotting data | `../data/figP01_*.tsv.gz`; sources and sha256 in `../data/phage_provenance.json` |
| produced by | `manuscript/prep/phage_prepare_data.py` (copies values; nothing re-estimated) |
| upstream: (a) | `Validation/mutual_information/tables/{mi_single_and_pairs,conditional_mi_given_length}.tsv` (genome_entropy #108), from `scripts/analyze_phage_mi.py` |
| upstream: (b, c) | `Validation/Inphared/tables/{model_metrics_pooled,evidence_rates}.tsv` (#110), from `scripts/03_models.py`, `07_evidence.py` |
| upstream: (d) | Setonix `/scratch/pawsey1018/edwa0468/genome_entropy/work/phage_rank/` (`tables/{orfs,cds}.parquet`, `rank_orfs_{relabelled_intrinsic,orig}/`), the corrected ranking run of #110 |
| configuration | MI set: genome_entropy ≈0.2.0, modernprost-50M, encoding size 20,000, Setonix ROCm fp16. Ranking set: ORFs from v0.1.15, relabelled with the current matcher; same model and encoding size. Effective ORF floor 90 aa (#103) for both. |
| superseded inputs not used | legacy `in_genbank` labels (MI 0.383 bits; 52,057 candidates / 1,797 replacements) |

## Data dictionary

| file | columns used | meaning |
|---|---|---|
| `figP01_mutual_information` | `label`, `mi_balanced_bits`, `boot_lo_bits`, `boot_hi_bits`, `fraction_of_I_X_retained_given_len20` | MI (bits) with the label on a class-balanced sample (H = 1 bit); share of the full-population MI retained given length |
| `figP01_ranking_models` | `set`, `auc_relabelled`, `ap_relabelled` | pooled out-of-fold metrics on corrected labels |
| `figP01_foldseek_rates` | `arm`, `db` (`any`, `non_bfvd`), `fullcov_rate`, `fullcov_ci_lo/hi`, `queries` | full-length hit rate |
| `figP01_om869708_orfs` | `g_start`, `g_end` (0-based, end-exclusive), `strand`, `frame`, `probability_corrected`, `matches_cds`, `replacement_rank` | ORFs ≥90 aa |
| `figP01_om869708_cds` | `cds_id`, `g_start`, `g_end`, `strand`, `product` | annotated CDS |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors, cm
from matplotlib.patches import FancyArrow

import msfig

msfig.style()
SLUG = "figP01_phage"

In [ ]:
mi = msfig.data("figP01_mutual_information.tsv.gz")
models = msfig.data("figP01_ranking_models.tsv.gz").set_index("set")
fs = msfig.data("figP01_foldseek_rates.tsv.gz")
orfs = msfig.data("figP01_om869708_orfs.tsv.gz")
cds = msfig.data("figP01_om869708_cds.tsv.gz")

# The numbers the caption and text quote.
three_di = mi.set_index("features").loc["three_di_entropy"]
assert round(three_di.mi_balanced_bits, 3) == 0.587
assert round(mi.set_index("features").loc["protein_entropy+three_di_entropy"].mi_balanced_bits, 3) == 0.644
assert round(three_di.fraction_of_I_X_retained_given_len20, 2) == 0.82
assert round(models.loc["intrinsic", "auc_relabelled"], 3) == 0.976
assert round(models.loc["length_only", "auc_relabelled"], 3) == 0.733
assert round(models.loc["orig14_stored", "auc_relabelled"], 3) == 0.740
rate = fs.set_index(["arm", "db"]).fullcov_rate
assert round(rate["candidate", "any"], 3) == 0.338 and round(rate["candidate", "non_bfvd"], 3) == 0.061
o3 = orfs.set_index("orf_id").loc["orf3"]; o7 = orfs.set_index("orf_id").loc["orf7"]
assert round(o3.probability_corrected, 3) == 0.889 and round(o7.probability_corrected, 3) == 0.175
assert o3.replacement_rank == 179 and o3.replacement_pairs_total == 4102

In [ ]:
fig = plt.figure(figsize=(7.2, 6.4))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.15], height_ratios=[1, 1.05],
                      hspace=0.55, wspace=0.65)
C = msfig.PALETTE

# (a) mutual information --------------------------------------------------
ax = fig.add_subplot(gs[0, 0])
order = ["three_di_entropy", "three_di_twelve_state_mutual_information",
         "protein_entropy", "twelve_state_entropy", "protein_length",
         "dna_entropy"]
d = mi.set_index("features").loc[order]
y = np.arange(len(d))[::-1] + 1
ax.barh(y, d.mi_balanced_bits, color=C["blue"], height=0.62,
        xerr=[d.mi_balanced_bits - d.boot_lo_bits, d.boot_hi_bits - d.mi_balanced_bits],
        error_kw=dict(lw=0.8, capsize=1.5))
pair = mi.set_index("features").loc["protein_entropy+three_di_entropy"]
ax.barh([len(d) + 1.2], [pair.mi_balanced_bits], color=C["sky"], height=0.62)
ax.set_yticks(list(y) + [len(d) + 1.2], list(d.label) + ["protein + 3Di entropy"])
for yy, (v, f) in zip(y, zip(d.mi_balanced_bits, d.fraction_of_I_X_retained_given_len20)):
    if not np.isnan(f) and f <= 1:
        ax.text(v + 0.015, yy, f"{f:.0%}", va="center", fontsize=6.5, color=C["black"])
ax.text(pair.mi_balanced_bits + 0.015, len(d) + 1.2, f"+{pair.mi_balanced_bits - d.mi_balanced_bits.iloc[0]:.3f} bits",
        va="center", fontsize=6.5)
ax.set_xlim(0, 0.8)
ax.set_xlabel("mutual information with label (bits)")
ax.set_title("a  information about annotation", loc="left", fontweight="bold")
ax.text(0.98, 0.02, "% = share retained\ngiven length", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=6, color=C["grey"])

# (b) ranking models ----------------------------------------------------
ax = fig.add_subplot(gs[0, 1])
sets = [("length_only", "length"), ("length_3di", "length + 3Di"),
        ("entropy_only", "entropies"), ("intrinsic", "entropies + lengths"),
        ("orig14_stored", "legacy model\n(strand shortcut)")]
x = np.arange(len(sets))
w = 0.38
auc = [models.loc[s, "auc_relabelled"] for s, _ in sets]
ap = [models.loc[s, "ap_relabelled"] for s, _ in sets]
ax.bar(x - w / 2, auc, w, label="ROC AUC", color=C["blue"])
ax.bar(x + w / 2, ap, w, label="average precision", color=C["orange"])
for xx, v in zip(x, auc):
    ax.text(xx - w / 2, v + 0.01, f"{v:.2f}", ha="center", fontsize=6)
ax.axhline(0.397, color=C["grey"], lw=0.7, ls=":")  # prevalence = AP of a random ranking
ax.set_xticks(x, [l for _, l in sets], rotation=30, ha="right")
ax.set_ylim(0, 1.25)
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_ylabel("score on corrected labels")
ax.legend(loc="upper center", ncol=2, fontsize=6.5, bbox_to_anchor=(0.5, 1.02))
ax.set_title("b  ranking ORFs as genes", loc="left", fontweight="bold")

# (c) Foldseek validation -----------------------------------------------
ax = fig.add_subplot(gs[1, 0])
arms = [("annotated", "annotated CDS"), ("replacement", "proposed replacements"),
        ("candidate", "candidates"), ("replaced", "replaced annotated"),
        ("intergenic_lo", "intergenic (low score)"), ("frameshift_lo", "frameshift (low)"),
        ("antisense_lo", "antisense (low)"), ("candidate_shuffled_3di", "candidates, 3Di shuffled")]
r = fs.set_index(["arm", "db"])
y = np.arange(len(arms))[::-1]
for k, (db, lbl, col) in enumerate([("any", "4 databases", C["blue"]),
                                     ("non_bfvd", "excluding viral (BFVD)", C["vermillion"])]):
    v = np.array([r.loc[(a, db), "fullcov_rate"] for a, _ in arms]) * 100
    lo = np.array([r.loc[(a, db), "fullcov_ci_lo"] for a, _ in arms]) * 100
    hi = np.array([r.loc[(a, db), "fullcov_ci_hi"] for a, _ in arms]) * 100
    yy = y + (0.17 if k == 0 else -0.17)
    shown = np.maximum(v, 0.005)  # zero has no place on a log axis
    ax.errorbar(shown, yy, xerr=[shown - np.maximum(lo, 0.005), np.maximum(hi, 0.005) - shown],
                fmt="o", ms=3.2, color=col, lw=0.8, capsize=1.5, label=lbl)
ax.set_xscale("log")
ax.set_xlim(0.004, 100)
ax.set_yticks(y, [l for _, l in arms])
ax.set_xlabel("full-length structural hits (%)")
ax.legend(loc="lower right", fontsize=6)
ax.set_title("c  Foldseek validation", loc="left", fontweight="bold")

# (d) OM869708 locus ------------------------------------------------------
ax = fig.add_subplot(gs[1, 1])
L = int(orfs.record_length.iloc[0])
cmap = plt.get_cmap("cividis")
norm = colors.Normalize(0, 1)
lanes = {("+", 1): 5, ("+", 2): 4, ("+", 3): 3, ("-", 1): 1, ("-", 2): 0, ("-", 3): -1}
def arrow(ax, s, e, strand, yc, fc, ec, lw=0.6, h=0.62):
    length = e - s
    head = min(120, length * 0.35)
    if strand == "+":
        ax.add_patch(FancyArrow(s, yc, length, 0, width=h, head_width=h, head_length=head,
                                length_includes_head=True, fc=fc, ec=ec, lw=lw))
    else:
        ax.add_patch(FancyArrow(e, yc, -length, 0, width=h, head_width=h, head_length=head,
                                length_includes_head=True, fc=fc, ec=ec, lw=lw))
for c in cds.itertuples():
    arrow(ax, c.g_start, c.g_end, c.strand, 6.4, "#d9d9d9", "#7f7f7f")
for o in orfs.itertuples():
    yc = lanes[(o.strand, int(o.frame))]
    arrow(ax, o.g_start, o.g_end, o.strand, yc, cmap(norm(o.probability_corrected)),
          "black" if o.matches_cds else "none", lw=0.9)
# Label ORF3 above its arrow and ORF7 below (lane +3 is empty there), both
# right-aligned to the arrow ends so nothing runs past the axis.
o = orfs.set_index("orf_id").loc["orf3"]
yc = lanes[(o.strand, int(o.frame))]
arrow(ax, o.g_start, o.g_end, o.strand, yc, cmap(norm(o.probability_corrected)), C["vermillion"], lw=1.4)
ax.text(o.g_end, yc + 0.45, f"ORF3  p = {o.probability_corrected:.2f}", ha="right", va="bottom",
        fontsize=6.3, color=C["vermillion"])
o = orfs.set_index("orf_id").loc["orf7"]
yc = lanes[(o.strand, int(o.frame))]
ax.text(o.g_end, yc - 0.45, f"ORF7  p = {o.probability_corrected:.2f}", ha="right", va="top",
        fontsize=6.3)
ax.set_xlim(0, L)
ax.set_ylim(-1.8, 7.3)
ax.set_yticks([6.4, 5, 4, 3, 1, 0, -1], ["annotated CDS", "+1", "+2", "+3", "−1", "−2", "−3"])
ax.set_xlabel("position in OM869708 (bp)")
ax.grid(False)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
cb = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, fraction=0.04, pad=0.02)
cb.set_label("coding probability (corrected model)", fontsize=6.5)
cb.ax.tick_params(labelsize=6)
ax.set_title("d  an annotation error in OM869708", loc="left", fontweight="bold")

msfig.save(fig, SLUG)

## Notes

- (a) uses the class-balanced MI because the label is 40% positive; H(label) = 1 bit on the balanced sample. The length-conditioned share is computed on the full population (`conditional_mi_given_length.tsv`), where H(label) = 0.970 bits. DNA entropy's conditional share (>100%) is not shown: its marginal MI is ~0.003 bits, so the ratio is meaningless.
- (b) "legacy model" is the original 14-feature model's stored out-of-fold predictions, scored against corrected labels; its training labels had 3 of 2.9 M minus-strand ORFs positive and `strand_plus` carried 0.89 of its gain (`figPS01`).
- (c) shows a floor of 0.005% for zero counts (shuffled 3Di, excluding BFVD: 0 of 20,000).
- (d) ORF coordinates are stop-to-stop; ORF7 is the ORF matching CDS_0007. The BLASTP evidence for ORF3 is in `figP01_om869708_blastp.tsv.gz` once committed (see decisions D7).